# ActiTect / RBDisco Tutorial on Public Dataset

Here we demonstrate the usage of our codebase the publicly available dataset of the Newcastle cohort ([Van Hees et al.](https://zenodo.org/records/1160410)). It consists of one-night GENEActiv recordings from 26 patients, including 3 RBD cases. As elaborated in our publication, this dataset serves as an out-of-domain test, as the model trained on Axivity AX6 data is evaluated on GENEActiv recordings. In addition, the dataset is very small, comprising only a single night per subject (with some nights only having partial sleep coverage), which does not capture night-to-night variability, and includes only a few positive RBD cases; we refer to the main manuscript for a detailed discussion of the resulting performance and its interpretation.

In this tutorial we are using the Python API of ActiTect and RBDisco. You can also use the command line interfaces as described in the [main README](../../README.md). Before staring this tutorial, make sure you've correctly installed ActiTect/ RBDisco (also described in README).

---
## 1. Download the Data & Setup Paths

The dataset is available on Zenodo at [https://zenodo.org/records/1160410](https://zenodo.org/records/1160410). You can manually download it, unzip it and update the Path pointing to the data directory below.

In [1]:
from pathlib import Path

from actitect.utils import get_experiment_root

DATA_DIR: Path = get_experiment_root() / 'data/dataset_psgnewcastle2015_v1.0' # OVERWRITE to your download dir or move unzipped dir here
ACC_DATA_ROOT: Path = DATA_DIR / 'acc'
META_FILE: Path = DATA_DIR / 'participants_info.csv'

for _p  in (DATA_DIR, ACC_DATA_ROOT, META_FILE):
    assert _p.exists(), f"{_p} does not exist"

OUT_DIR = Path.cwd()  / 'out'  # change if needed


---
## 2. Run ActiTect / RBDisco

We simply loop over the actigraphy files and run `actitect.api.load()`, `actitect.api.process()` and `actitect.api.compute_sleep_motor_features()` to preprocess the data and extract the sleep motion features. A RBD prediction based on these features is made with `actitect.rbdiso.predict()`.

In [2]:
import re
import pandas as pd
from typing import Iterator

from actitect import utils
import logging

from actitect.api import load, process, compute_sleep_motor_features
from actitect.rbdisco import predict

logger = logging.getLogger(__name__)

# helpers
## read newcastle meta information
def read_meta(csv_file: Path, *, diag_col: str = 'Diagnosis') -> pd.DataFrame:
    df = pd.read_csv(csv_file, index_col=0)
    df['rbd_gt'] = infer_rbd_gt_from_diagnosis(df.get(diag_col, pd.Series(index=df.index, dtype='object')))
    return df

## file iterator for actigraphy recordings
def iterate_files(data_root_dir: Path, meta_df: pd.DataFrame, *, ext: str = '*.bin') -> Iterator[Path]:

    def _parse_filename(_fname: str, pattern=None) -> dict:
        if pattern is None:
            pattern = re.compile(
                r"""
                ^MECSLEEP
                (?P<id>\d{2})
                _(?P<side>left|right)\s+wrist
                _(?P<code>\d+)
                _(?P<ts>\d{4}-\d{2}-\d{2}\s+\d{2}-\d{2}-\d{2})
                (?:\.bin)?$
                """,
                re.VERBOSE
            )
        elif isinstance(pattern, str):
            pattern = re.compile(pattern, re.VERBOSE)

        m = pattern.match(_fname)
        if not m:
            raise ValueError(f"Filename does not match expected format: {_fname}")

        out = m.groupdict()
        out["id"] = int(out["id"])
        out["timestamp"] = pd.to_datetime(out.pop("ts"), format="%Y-%m-%d %H-%M-%S")
        return out

    all_files = sorted(list(data_root_dir.glob(ext)))
    assert len(all_files) > 0, f"{ext} not found in {data_root_dir}"
    logger.info(f"Found {len(all_files)} files in {data_root_dir}")

    for file_path in all_files:
        fname = file_path.stem
        metadata = _parse_filename(fname)
        _id = metadata['id']
        metadata.update(meta_df.loc[_id].to_dict())
        yield Path(file_path), metadata

## map newcastle diagnosis to binary RBD label
def infer_rbd_gt_from_diagnosis(diagnosis: pd.Series) -> pd.Series:
    s = diagnosis.fillna("").astype(str).str.lower()
    return s.str.contains(r"\brbd\b|rem\s*sleep\s*behaviou?r\s*disorder", regex=True).astype(int)

# main loop: load -> preprocess -> calculate features -> predict RBD
def run_actitect_rbdisco() -> None:
    utils.setup_logging()
    utils.check_make_dir(OUT_DIR, use_existing=True)
    meta_df = read_meta(META_FILE)

    pred_rows = []

    with utils.custom_tqdm(total=len(sorted(ACC_DATA_ROOT.glob('*.bin')))) as pbar:
        pbar.set_description(f"[PROGRESS] Processing files")

        for file_path, file_info in iterate_files(ACC_DATA_ROOT, meta_df, ext='*.bin'):
            _id, _side = file_info['id'], file_info['side']

            if (int(_id), str(_side).strip().lower()) in {(31, 'right'), (50, 'left'), (38, 'left'), (39, 'right'), (42, 'left'), (42, 'right'), (50, 'left'), (50, 'right')}:
                logger.warning(f"Skipping {_id}-{_side}: No nights found")
                continue  # some cases have incomplete sleep coverage, leading to failed sleep detection, for the demo we skip them early

            pbar.set_postfix({'id': _id, 'side': _side, 'status': 'prepro.'})

            out_dir = utils.check_make_dir(OUT_DIR / f"ID{_id}" / _side, use_existing=True)
            feat_file = out_dir / f"features_{_id}_{_side}.csv"

            logger.info("Processing %s (%s, %s)", file_path.name, _id, _side)

            if feat_file.is_file():  # resume behaviour
                feat_df = pd.read_csv(feat_file)

            else:  # full
                raw_df, _ = load(file_path, subject_id=_id)
                processed_df, _ = process(raw_df, subject_id=_id)

                pbar.set_postfix({'id': _id, 'side': _side, 'status': 'features'})
                feat_df = compute_sleep_motor_features(processed_df, subject_id=_id)

                if feat_df.empty:
                    logger.warning("No valid nights for %s", file_path.name)
                    continue

                feat_df.to_csv(feat_file, index=False)

            pbar.set_postfix({'id': _id, 'side': _side, 'status': 'prediction'})
            pred_df = predict(feat_df, model='multiCenterCore', subject_id=_id).copy().to_dict('records')[0]

            pred_rows.append(dict(
                id=_id,
                side=_side,
                total_nights=pred_df['n_total_nights'],
                diagnosis=file_info['Diagnosis'],
                rbd_gt=file_info['rbd_gt'],
                proba_rbd=pred_df['mean_prob_per_night'],
            ))

            pbar.update(1)

    _df = pd.DataFrame(pred_rows)
    subj_df = (_df.groupby('id', as_index=False).agg(
            diagnosis=('diagnosis', 'first'),
            rbd_gt=('rbd_gt', 'max'),
            proba_rbd=('proba_rbd', 'mean')))

    subj_df['rbd_pred'] = (subj_df['proba_rbd'] >= .5).astype(int)
    subj_df.drop(columns=['proba_rbd'], inplace=True)
    return subj_df

pred_df = run_actitect_rbdisco()

 (15:08:59) - [actitect.utils.io_utils|INFO]: (io): dir '/Users/david/Desktop/code/actitect_dev/actitect/examples/public_dataset/out' exists and will be used.Set 'use_existing' to False, to create new dir to avoid potential overwriting.


 (15:08:59) - [__main__|INFO]: Found 55 files in /Users/david/Desktop/code/actitect_dev/data/dataset_psgnewcastle2015_v1.0/acc


[PROGRESS] Processing files:   0%|□□□□□□□□□□| 0/55 [00:00<?, ?it/s, id=1, side=left, status=prepro.]

 (15:08:59) - [actitect.utils.io_utils|INFO]: (io): dir '/Users/david/Desktop/code/actitect_dev/actitect/examples/public_dataset/out/ID1/left' exists and will be used.Set 'use_existing' to False, to create new dir to avoid potential overwriting.


 (15:08:59) - [__main__|INFO]: Processing MECSLEEP01_left wrist_012870_2013-06-12 11-40-37.bin (1, left)


...

 (15:09:01) - [actitect.utils.io_utils|INFO]: (io): dir '/Users/david/Desktop/code/actitect_dev/actitect/examples/public_dataset/out/ID60/right' exists and will be used.Set 'use_existing' to False, to create new dir to avoid potential overwriting.


 (15:09:01) - [__main__|INFO]: Processing MECSLEEP60_right wrist_018141_2015-05-21 15-32-02.bin (60, right)


[PROGRESS] Processing files:  85%|■■■■■■■■□□| 47/55 [00:02<00:00, 22.61it/s, id=60, side=right, status=prediction]

 (15:09:01) - [actitect.rbdisco.core.types|INFO]: applying pre-fitted global robust scaler.


[PROGRESS] Processing files:  87%|■■■■■■■■□□| 48/55 [00:02<00:00, 22.31it/s, id=60, side=right, status=prediction]


---
## 3. Summarize Results
Due to limited statistical power, we report only counts of true and false positives and negatives. Refer to the manuscript for a full discussion.

In [3]:
tp = ((pred_df['rbd_gt'] == 1) & (pred_df['rbd_pred'] == 1)).sum()
fp = ((pred_df['rbd_gt'] == 0) & (pred_df['rbd_pred'] == 1)).sum()
tn = ((pred_df['rbd_gt'] == 0) & (pred_df['rbd_pred'] == 0)).sum()
fn = ((pred_df['rbd_gt'] == 1) & (pred_df['rbd_pred'] == 0)).sum()

logger.info(f"RBDisco prediction summary: \n \t ↳ TP: {tp:2.0f}, FP: {fp:2.0f} \n \t ↳ FN: {fn:2.0f}, TN: {tn:2.0f}")

 (15:09:02) - [__main__|INFO]: RBDisco prediction summary: 
 	 ↳ TP:  2, FP:  5 
 	 ↳ FN:  1, TN: 18
